# Importing libs

In [186]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

# Initial DF profile

In [140]:
df = pd.read_csv(
    r"/home/gusdpaula/code-dev/MLENG_FIAP/fase_4/data/accepted_2007_to_2018Q4.csv"
)

/tmp/ipykernel_107449/1279439524.py:1: DtypeWarning: Columns (0: id, 19: desc, 49: next_pymnt_d, 59: verification_status_joint, 118: sec_app_earliest_cr_line, 129: hardship_type, 130: hardship_reason, 131: hardship_status, 134: hardship_start_date, 135: hardship_end_date, 136: payment_plan_start_date, 139: hardship_loan_status, 145: debt_settlement_flag_date, 146: settlement_status, 147: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [141]:
df

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260696,88985880,NaN,40000.0,40000.0,40000.0,60 months,10.49,859.56,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260697,88224441,NaN,24000.0,24000.0,24000.0,60 months,14.49,564.56,C,C4,...,NaN,NaN,Cash,Y,Mar-2019,ACTIVE,Mar-2019,10000.0,44.82,1.0
2260698,88215728,NaN,14000.0,14000.0,14000.0,60 months,14.49,329.33,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260699,Total amount funded in policy code 1: 1465324575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [142]:
df.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
       'disbursement_method', 'debt_settlement_flag',
       'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
       'settlement_amount', 'settlement_percentage', 'settlement_term'],
      dtype='str', length=151)

In [143]:
df["loan_status"].value_counts(normalize=True)

loan_status
Fully Paid                                             0.476298
Current                                                0.388521
Charged Off                                            0.118796
Late (31-120 days)                                     0.009496
In Grace Period                                        0.003732
Late (16-30 days)                                      0.001924
Does not meet the credit policy. Status:Fully Paid     0.000879
Does not meet the credit policy. Status:Charged Off    0.000337
Default                                                0.000018
Name: proportion, dtype: float64

# Profilling cols

In [144]:
def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Profile each column in a DataFrame.

    Returns a summary with data types, missing values,
    cardinality, statistics, and frequent values.
    """

    profiles = []

    for col in df.columns:
        s = df[col]
        non_null = s.dropna()

        profile = {
            "column": col,
            "dtype": str(s.dtype),
            "total_rows": len(s),
            "missing_count": s.isna().sum(),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "unique_count": s.nunique(),
            "unique_pct": round(s.nunique() / len(s) * 100, 2) if len(s) else 0,
            "memory_kb": round(s.memory_usage(deep=True) / 1024, 2),
        }

        if pd.api.types.is_numeric_dtype(s):
            profile.update(
                {
                    "mean": non_null.mean(),
                    "std": non_null.std(),
                    "min": non_null.min(),
                    "p25": non_null.quantile(0.25),
                    "median": non_null.median(),
                    "p75": non_null.quantile(0.75),
                    "max": non_null.max(),
                    "skewness": non_null.skew(),
                    "zero_count": (s == 0).sum(),
                    "negative_count": (s < 0).sum(),
                }
            )

            # IQR-based outlier detection
            q1 = non_null.quantile(0.25)
            q3 = non_null.quantile(0.75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            profile["outlier_count"] = ((s < lower_bound) | (s > upper_bound)).sum()

        else:
            profile.update(
                {
                    "mean": np.nan,
                    "std": np.nan,
                    "min": np.nan,
                    "p25": np.nan,
                    "median": np.nan,
                    "p75": np.nan,
                    "max": np.nan,
                    "skewness": np.nan,
                    "zero_count": np.nan,
                    "negative_count": np.nan,
                    "outlier_count": np.nan,
                }
            )

        # Most frequent values, including missing values
        profile["top_values"] = s.value_counts(dropna=False).head(5).to_dict()

        profiles.append(profile)

    return pd.DataFrame(profiles).set_index("column")

In [145]:
profile_df = profile_dataframe(df)

In [146]:
profile_df

,dtype,total_rows,missing_count,missing_pct,unique_count,unique_pct,memory_kb,mean,std,min,p25,median,p75,max,skewness,zero_count,negative_count,outlier_count,top_values
column,,,,,,,,,,,,,,,,,,,
id,object,2260701,0,0.00,2260701,100.00,80828.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{68407277: 1, 68355089: 1, 68341763: 1, 663107..."
member_id,float64,2260701,2260701,100.00,0,0.00,17661.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,{nan: 2260701}
loan_amnt,float64,2260701,33,0.00,1572,0.07,17661.86,15046.931228,9190.245488,500.00,8000.0,12900.00,20000.0000,40000.00,0.777782,0.0,0.0,35215.0,"{10000.0: 187236, 20000.0: 131006, 15000.0: 12..."
funded_amnt,float64,2260701,33,0.00,1572,0.07,17661.86,15041.664057,9188.413022,500.00,8000.0,12875.00,20000.0000,40000.00,0.778779,0.0,0.0,35215.0,"{10000.0: 187146, 20000.0: 130816, 15000.0: 12..."
funded_amnt_inv,float64,2260701,33,0.00,10057,0.44,17661.86,15023.437745,9192.331679,0.00,8000.0,12800.00,20000.0000,40000.00,0.778254,233.0,0.0,35212.0,"{10000.0: 177561, 20000.0: 120453, 15000.0: 11..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
settlement_status,str,2260701,2226455,98.49,3,0.00,71444.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{nan: 2226455, 'ACTIVE': 14704, 'COMPLETE': 14..."
settlement_date,str,2260701,2226455,98.49,90,0.00,71483.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{nan: 2226455, 'Jan-2019': 1710, 'Oct-2018': 1..."
settlement_amount,float64,2260701,2226455,98.49,21941,0.97,17661.86,5010.664267,3693.122590,44.21,2208.0,4146.11,6850.1725,33601.00,1.311380,0.0,0.0,1054.0,"{nan: 2226455, 5000.0: 65, 4000.0: 52, 6000.0:..."


In [147]:
profile_df[profile_df["missing_pct"] > 20].sort_values(
    by="missing_pct", ascending=False
).shape

(58, 19)

In [148]:
df = df.drop(columns=profile_df[profile_df["missing_pct"] > 20].index)

In [149]:
df.shape

(2260701, 93)

# Feature Engineering

In [150]:
df.loan_status.value_counts(normalize=True)

loan_status
Fully Paid                                             0.476298
Current                                                0.388521
Charged Off                                            0.118796
Late (31-120 days)                                     0.009496
In Grace Period                                        0.003732
Late (16-30 days)                                      0.001924
Does not meet the credit policy. Status:Fully Paid     0.000879
Does not meet the credit policy. Status:Charged Off    0.000337
Default                                                0.000018
Name: proportion, dtype: float64

In [151]:
df.loan_status = df.loan_status.map(
    {
        "Fully Paid": 0,
        "Charged Off": 1,
        "Default": 1,
        "Late (31-120 days)": 1,
        "Late (16-30 days)": 1,
        "In Grace Period": 1,
        "Current": 0,
        "Does not meet the credit policy. Status:Fully Paid": 0,
        "Does not meet the credit policy. Status:Charged Off": 1,
    }
)

In [152]:
df.loan_status.value_counts(normalize=True)

loan_status
0.0    0.865698
1.0    0.134302
Name: proportion, dtype: float64

In [153]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 93 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   id                          object 
 1   loan_amnt                   float64
 2   funded_amnt                 float64
 3   funded_amnt_inv             float64
 4   term                        str    
 5   int_rate                    float64
 6   installment                 float64
 7   grade                       str    
 8   sub_grade                   str    
 9   emp_title                   str    
 10  emp_length                  str    
 11  home_ownership              str    
 12  annual_inc                  float64
 13  verification_status         str    
 14  issue_d                     str    
 15  loan_status                 float64
 16  pymnt_plan                  str    
 17  url                         str    
 18  purpose                     str    
 19  title                       str 

In [154]:
del df["id"]

In [155]:
profile_df[(profile_df["missing_pct"] <= 20) & (profile_df["dtype"] == "str")]

,dtype,total_rows,missing_count,missing_pct,unique_count,unique_pct,memory_kb,mean,std,min,p25,median,p75,max,skewness,zero_count,negative_count,outlier_count,top_values
column,,,,,,,,,,,,,,,,,,,
term,str,2260701,33,0.00,2,0.00,130254.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{' 36 months': 1609754, ' 60 months': 650914, ..."
grade,str,2260701,33,0.00,7,0.00,110385.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'B': 663557, 'C': 650053, 'A': 433027, 'D': 3..."
sub_grade,str,2260701,33,0.00,35,0.00,112593.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'C1': 145903, 'B5': 140288, 'B4': 139793, 'B3..."
emp_title,str,2260701,167002,7.39,512694,22.68,137300.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{nan: 167002, 'Teacher': 38824, 'Manager': 342..."
emp_length,str,2260701,146940,6.50,11,0.00,121689.87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'10+ years': 748005, '2 years': 203677, '< 1 ..."
home_ownership,str,2260701,33,0.00,6,0.00,121102.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'MORTGAGE': 1111450, 'RENT': 894929, 'OWN': 2..."
verification_status,str,2260701,33,0.00,3,0.00,134806.74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'Source Verified': 886231, 'Not Verified': 74..."
issue_d,str,2260701,33,0.00,139,0.01,125839.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'Mar-2016': 61992, 'Oct-2015': 48631, 'May-20..."
loan_status,str,2260701,33,0.00,9,0.00,128290.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'Fully Paid': 1076751, 'Current': 878317, 'Ch..."


In [156]:
cols_to_drop = profile_df.loc[
    (profile_df["dtype"] == "str") & (profile_df["unique_pct"] > 20)
].index

df = df.drop(columns=cols_to_drop, errors="ignore")

In [160]:
del df["title"]

In [ ]:
"""
term - onehot
grade - rank
sub_grade - rank
emp_length - rank
home_ownership - onehot
verification_status - onehot
issue_d - date
pymnt_plan - binary
purpose - onehot
zip_code - numeric
addr_state - onehot
earliest_cr_line - date
initial_list_status - onehot
last_pymnt_d - date
disbursement_method - onehot
debt_settlement_flag - binary
"""

'\nterm - onehot\ngrade - rank\nsub_grade - rank\nemp_length - rank\nhome_ownership - onehot\nverification_status - onehot\nissue_d - date\npymnt_plan - binary\n'

In [161]:
df.select_dtypes(include=["object", "string"]).columns

Index(['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
       'verification_status', 'issue_d', 'pymnt_plan', 'purpose', 'zip_code',
       'addr_state', 'earliest_cr_line', 'initial_list_status', 'last_pymnt_d',
       'last_credit_pull_d', 'application_type', 'hardship_flag',
       'disbursement_method', 'debt_settlement_flag'],
      dtype='str')

In [167]:
df.debt_settlement_flag.value_counts(normalize=True)

debt_settlement_flag
N    0.984851
Y    0.015149
Name: proportion, dtype: float64

In [191]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Drop identifiers and uninformative columns
    df = df.drop(columns=["id", "member_id", "url"], errors="ignore")

    # 2. Rank encoding
    rank_columns = {
        "grade": list("ABCDEFG"),
        "sub_grade": [f"{grade}{i}" for grade in "ABCDEFG" for i in range(1, 6)],
        "emp_length": [
            "< 1 year",
            "1 year",
            "2 years",
            "3 years",
            "4 years",
            "5 years",
            "6 years",
            "7 years",
            "8 years",
            "9 years",
            "10+ years",
        ],
    }

    for col, categories in rank_columns.items():
        if col in df.columns:
            mapping = {value: rank for rank, value in enumerate(categories)}
            df[col] = df[col].map(mapping)

    # 3. Binary encoding
    binary_columns = [
        "pymnt_plan",
        "debt_settlement_flag",
        "hardship_flag",
    ]

    binary_mapping = {
        "n": 0,
        "y": 1,
        "no": 0,
        "yes": 1,
    }

    for col in binary_columns:
        if col in df.columns:
            df[col] = (
                df[col].astype("string").str.strip().str.lower().map(binary_mapping)
            )

    # 4. Date encoding
    date_columns = [
        "issue_d",
        "earliest_cr_line",
        "last_pymnt_d",
        "last_credit_pull_d",
        "next_pymnt_d",
        "sec_app_earliest_cr_line",
        "hardship_start_date",
        "hardship_end_date",
        "payment_plan_start_date",
        "debt_settlement_flag_date",
        "settlement_date",
    ]

    today = pd.Timestamp.today().normalize()

    for col in date_columns:
        if col in df.columns:
            dates = pd.to_datetime(df[col], format="%b-%Y", errors="coerce")

            df[f"{col}_days_from_today"] = (today - dates).dt.days

            df = df.drop(columns=col)

    # 5. ZIP code: extract digits
    if "zip_code" in df.columns:
        df["zip_code"] = (
            df["zip_code"].astype("string").str.extract(r"(\d+)", expand=False)
        )

        df["zip_code"] = pd.to_numeric(df["zip_code"], errors="coerce")

    # 6. One-hot encoding
    onehot_columns = [
        "term",
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "disbursement_method",
        "application_type",
    ]

    existing_columns = [col for col in onehot_columns if col in df.columns]

    df = pd.get_dummies(df, columns=existing_columns, dummy_na=False, dtype=int)

    # 7. Drop columns that are entirely missing
    df = df.dropna(axis=1, how="all")

    return df

In [192]:
df_eng = feature_engineering(df)

In [193]:
df_eng

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,emp_length,annual_inc,loan_status,...,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY,initial_list_status_f,initial_list_status_w,disbursement_method_Cash,disbursement_method_DirectPay,application_type_Individual,application_type_Joint App
0,3600.0,3600.0,3600.0,13.99,123.03,2,13,10.0,55000.0,0.0,...,0,0,0,0,0,1,1,0,1,0
1,24700.0,24700.0,24700.0,11.99,820.28,2,10,10.0,65000.0,0.0,...,0,0,0,0,0,1,1,0,1,0
2,20000.0,20000.0,20000.0,10.78,432.66,1,8,10.0,63000.0,0.0,...,0,0,0,0,0,1,1,0,0,1
4,10400.0,10400.0,10400.0,22.45,289.91,5,25,3.0,104433.0,0.0,...,0,0,0,0,0,1,1,0,1,0
6,20000.0,20000.0,20000.0,9.17,637.58,1,6,10.0,180000.0,0.0,...,0,0,0,0,1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260693,24000.0,24000.0,24000.0,10.49,515.74,1,7,4.0,125000.0,0.0,...,0,0,0,0,1,0,1,0,1,0
2260695,24000.0,24000.0,24000.0,10.49,515.74,1,7,10.0,108000.0,0.0,...,0,0,0,0,1,0,1,0,1,0
2260696,40000.0,40000.0,40000.0,10.49,859.56,1,7,9.0,227000.0,0.0,...,0,0,0,0,1,0,1,0,1,0
2260697,24000.0,24000.0,24000.0,14.49,564.56,2,13,6.0,110000.0,1.0,...,0,0,0,0,1,0,1,0,1,0


In [194]:
# Identify float columns
float_cols = df_eng.select_dtypes(include="float").columns

# Drop rows with NaN in any float column
df_eng = df_eng.dropna(subset=float_cols)

In [195]:
df_eng

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,emp_length,annual_inc,loan_status,...,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY,initial_list_status_f,initial_list_status_w,disbursement_method_Cash,disbursement_method_DirectPay,application_type_Individual,application_type_Joint App
0,3600.0,3600.0,3600.0,13.99,123.03,2,13,10.0,55000.0,0.0,...,0,0,0,0,0,1,1,0,1,0
1,24700.0,24700.0,24700.0,11.99,820.28,2,10,10.0,65000.0,0.0,...,0,0,0,0,0,1,1,0,1,0
2,20000.0,20000.0,20000.0,10.78,432.66,1,8,10.0,63000.0,0.0,...,0,0,0,0,0,1,1,0,0,1
4,10400.0,10400.0,10400.0,22.45,289.91,5,25,3.0,104433.0,0.0,...,0,0,0,0,0,1,1,0,1,0
6,20000.0,20000.0,20000.0,9.17,637.58,1,6,10.0,180000.0,0.0,...,0,0,0,0,1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260693,24000.0,24000.0,24000.0,10.49,515.74,1,7,4.0,125000.0,0.0,...,0,0,0,0,1,0,1,0,1,0
2260695,24000.0,24000.0,24000.0,10.49,515.74,1,7,10.0,108000.0,0.0,...,0,0,0,0,1,0,1,0,1,0
2260696,40000.0,40000.0,40000.0,10.49,859.56,1,7,9.0,227000.0,0.0,...,0,0,0,0,1,0,1,0,1,0
2260697,24000.0,24000.0,24000.0,14.49,564.56,2,13,6.0,110000.0,1.0,...,0,0,0,0,1,0,1,0,1,0


In [196]:
df_eng.info()

<class 'pandas.DataFrame'>
Index: 1683229 entries, 0 to 2260698
Columns: 163 entries, loan_amnt to application_type_Joint App
dtypes: Int64(1), float64(73), int64(89)
memory usage: 2.1 GB


# Training the model

In [197]:
def split_and_scale(
    df: pd.DataFrame, target: str, test_size: float = 0.2, random_state: int = 42
):
    # Split features and target
    X = df.drop(columns=[target])
    y = df[target]

    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Select float columns only
    float_cols = X_train.select_dtypes(include=["float"]).columns

    # Scale using training data only
    scaler = StandardScaler()

    X_train.loc[:, float_cols] = scaler.fit_transform(X_train[float_cols])

    X_test.loc[:, float_cols] = scaler.transform(X_test[float_cols])

    return X_train, X_test, y_train, y_test, scaler

In [198]:
X_train, X_test, y_train, y_test, scaler = split_and_scale(df_eng, target="loan_status")

In [199]:
def train_logistic_regression(
    X_train, X_test, y_train, y_test, max_iter=1000, class_weight=None
):
    # Initialize model
    model = LogisticRegression(
        max_iter=max_iter, class_weight=class_weight, random_state=42
    )

    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Evaluation metrics
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }

    print("Evaluation Metrics")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model, metrics

In [200]:
# Check columns that are not numeric
non_numeric_cols = X_train.select_dtypes(exclude="number").columns

print("Non-numeric columns:")
print(X_train[non_numeric_cols].dtypes)

# Inspect some values
for col in non_numeric_cols:
    print(f"\n{col}:")
    print(X_train[col].dropna().unique()[:10])

Non-numeric columns:
Series([], dtype: object)


In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns

print("NaN values:", X_train.isna().sum().sum())
print("Infinite values:", np.isinf(X_train[numeric_cols]).sum().sum())

NaN values: 0
Infinite values: 0


In [204]:
numeric_cols = X_train.select_dtypes(include=np.number).columns

print("NaN values:", X_test.isna().sum().sum())
print("Infinite values:", np.isinf(X_test[numeric_cols]).sum().sum())

NaN values: 1
Infinite values: 0


In [205]:
# Identify columns with missing values in the test set
cols_with_nan = X_test.columns[X_test.isna().any()]

for col in cols_with_nan:
    X_test[col] = X_test[col].fillna(X_train[col].median())

In [206]:
numeric_cols = X_train.select_dtypes(include=np.number).columns

print("NaN values:", X_test.isna().sum().sum())
print("Infinite values:", np.isinf(X_test[numeric_cols]).sum().sum())

NaN values: 0
Infinite values: 0


In [207]:
model, metrics = train_logistic_regression(
    X_train, X_test, y_train, y_test, class_weight="balanced"
)

/home/gusdpaula/code-dev/MLENG_FIAP/fase_4/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Evaluation Metrics
accuracy: 0.9667
precision: 0.8357
recall: 0.9350
f1_score: 0.8826
roc_auc: 0.9882

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.97      0.98    291603
         1.0       0.84      0.93      0.88     45043

    accuracy                           0.97    336646
   macro avg       0.91      0.95      0.93    336646
weighted avg       0.97      0.97      0.97    336646


Confusion Matrix:
[[283325   8278]
 [  2930  42113]]
